In [ ]:
#附息债券的计算
import numpy_financial as npf
print(dir(npf))
print(len(dir(npf)))
interest_rate = npf.rate(nper=10, pmt=100, pv=-600,fv=1000)
interest_rate = npf.rate(10, 100, -600,1000)
print(f"计算出的期利率: {interest_rate:.4f}")
present_value = npf.pv(0.1, 1, 10, 100)
present_value = npf.pv(nper=1, rate=0.1, pmt=10, fv=100)
print(f"计算出的现值: {abs(present_value):.2f}")

In [ ]:
#年金的计算
c=100
r=0.15
n=10
g=0.02
# 1. 普通年金现值
def pv_annuity(c, r, n):
    return c * (1 - (1 + r)**-n) / r
#也可以用函数
present_value1=npf.pv(0.15,10,-100,0,when=0)
# 2. 普通年金终值
def fv_annuity(c, r, n):
    return c * ((1 + r)**n - 1) / r
future_value1=npf.fv(0.15,10,-100,0,when=0)
# 3. 预付年金现值
def pv_annuity_due(c, r, n):
    return pv_annuity(c, r, n) * (1 + r)
present_value2=npf.pv(0.15,10,-100,0,when=1)
# 4. 预付年金终值
def fv_annuity_due(c, r, n):
    return fv_annuity(c, r, n) * (1 + r)
future_value2=npf.fv(0.15,10,-100,0,when=1)
# 5. 增长年金现值（期末）
def pv_growing_annuity(c, r, g, n):
    if r == g:
        return c * n / (1 + r)
    return c / (r - g) * (1 - ((1 + g)/(1 + r))**n)
# 6. 增长年金终值（期末）
def fv_growing_annuity(c, r, g, n):
    if r == g:
        return c * n * (1 + r)**(n-1)
    return c * ((1 + r)**n - (1 + g)**n) / (r - g)
# 7. 预付增长年金现值
def pv_growing_annuity_due(c, r, g, n):
    return pv_growing_annuity(c, r, g, n) * (1 + r)
# 8. 预付增长年金终值
def fv_growing_annuity_due(c, r, g, n):
    return fv_growing_annuity(c, r, g, n) * (1 + r)
# 9. 普通永续年金
def pv_perpetuity(c, r):
    return c / r
# 10. 预付永续年金
def pv_perpetuity_due(c, r):
    return c / r * (1 + r)
# 11. 增长永续年金
def pv_growing_perpetuity(c, r, g):
    return c / (r - g)
# 12. 预付增长永续年金
def pv_growing_perpetuity_due(c, r, g):
    return c / (r - g) * (1 + r)
# 13. 递延永续年金（第k期末开始）
def pv_perpetuity_delayed(c, r, k):
    return (c / r) / (1 + r)**k
# 14. 递延增长永续年金
def pv_growing_perpetuity_delayed(c, r, g, k):
    return (c / (r - g)) / (1 + r)**k

In [ ]:
#投资项目净现值(NPV)与内部收益率(IRR)分析
import numpy as np
import matplotlib.pyplot as plt
cash_flow = [-100, 50, 60, 70, 100, 20]
discount_rate = 0.112
project_npv = npf.npv(discount_rate, cash_flow)
print(f"项目净现值 (NPV): {project_npv:.2f}")
irr = npf.irr(cash_flow)
print(f"内部收益率 (IRR): {irr:.4f} ({irr:.2%})")
# 绘制NPV随折现率变化的曲线，这段代码首先生成从0到50%的一组折现率，
#依次计算每个折现率下项目的净现值（NPV），并将折现率与对应NPV绘制成连续曲线；
#图中红色虚线为NPV=0的临界线，绿色虚线为项目内部收益率（IRR），代表NPV恰好为零时的折现率。
#整体图像清晰展示出项目净现值随折现率升高而单调递减的关系，当折现率低于IRR时NPV为正，项目具备投资价值，
#当折现率高于IRR时NPV为负，项目不具备投资价值，直观反映了折现率对项目可行性的影响。
discount_rates = np.linspace(0, 0.5, 100)  # 生成0到50%的折现率序列
npv_values = [npf.npv(r, cash_flow) for r in discount_rates]
plt.figure(figsize=(10, 6))
plt.plot(discount_rates, npv_values, 'b-', linewidth=2, label='NPV曲线')
plt.axhline(y=0, color='r', linestyle='--', alpha=0.7, linewidth=1.5, label='NPV = 0')
plt.axvline(x=irr, color='g', linestyle='--', alpha=0.7, linewidth=1.5, label=f'IRR = {irr:.2%}')
plt.xlabel('折现率', fontsize=12)
plt.ylabel('净现值 (NPV)', fontsize=12)
plt.title('项目净现值随折现率变化曲线', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
#股票数据获取与金融分析
import yfinance as yf
import scipy.stats as stats
df = yf.download('IBM', start='2023-01-01', end='2025-12-31')
# 1. 计算日收益率
df['daily_return'] = df['Close'].pct_change()   #日收益率 = 当日收盘价 / 前一日收盘价 − 1
ret = df['daily_return'].dropna()  # 去除缺失值
# 2. 收益率基本统计 
ret_mean = ret.mean()  # 日收益率均值
ret_std = ret.std()   # 日收益率波动（标准差）
# 3. 单样本t检验：收益是否显著不为0,p < 0.05 收益显著；p ≥ 0.05 收益不显著,股票是否具有稳定收益能力
t_stat, p_value = stats.ttest_1samp(ret, 0)
# 4. 年化收益率（按年度复利计算）
#prod()连乘，(1 + 日收益率).乘积 − 1
ret_yearly = ret.groupby(ret.index.year).apply(lambda x: (1 + x).prod() - 1) 
# 5. 月度收益率及统计指标 
ret_monthly = ret.groupby([ret.index.year, ret.index.month]).apply(lambda x: (1 + x).prod() - 1)
monthly_mean = ret_monthly.mean()    # 月均收益
monthly_std = ret_monthly.std()      # 月波动
monthly_max = ret_monthly.max()      # 最大月收益
monthly_min = ret_monthly.min()      # 最小月收益

In [ ]:
#收益率分布可视化呈现
import yfinance as yf
import pandas as pd
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt
from scipy.stats import norm

infile = 'http://datayyy.com/data_pickle/ff3Daily.pickle'
ff = pd.read_pickle(infile)
begdate='2020-1-1'
enddate='2021-12-31'
def ret_f(ticker, begdate, enddate):
    df = yf.download(ticker, start=begdate, end=enddate)
    ret = df["Close"].pct_change().dropna()
    return ret
a=ret_f('IBM',begdate,enddate)#得到ibm的日收益率
b=ret_f('MSFT',begdate,enddate)#得到msft的日收益率
#首先利用plt.hist函数绘制微软（MSFT）日收益率的频率直方图，并将纵轴标准化为概率密度形式，
#随后计算收益率序列的均值与标准差，基于这两个统计量拟合正态分布概率密度曲线并叠加在直方图上，
#通过图表直观展示实际收益率分布与理论正态分布的拟合程度，最终完成收益率分布可视化呈现。
[n, bins, patches] = plt.hist(b, 100, density=True)  #把收益率数据切成 100 个等宽区间
mu = b.mean()
sigma = b.std()
x = norm.pdf(bins, mu, sigma)
plt.plot(bins, x, color='red', lw=2)
plt.title("MSFT return dist")
plt.xlabel("return")
plt.ylabel("frequency")
plt.show()